# 05 — Peer Conversation Phase C (Issue 7)

Demonstrates the simultaneous peer-to-peer conversation phase:
1. Load data, create agents, assign exposure, build network
2. Run Phase C: citizens generate messages, then reflect on peer messages
3. Verify simultaneous update (messages generated BEFORE reflections)
4. Analyze peer conversation patterns and reflection content

**Covers:** Issue 7 (Peer Conversation Phase C)  
**Depends on:** Issues 5, 6

In [ ]:
import os, sys, random
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '../src')))

from cag.io.survey import load
from cag.abm.agent import SurveyedCitizen, PoliticalAgent
from cag.abm.environment import SurveyedNation
from cag.abm.attributes.opinion import ClimatePolicyID, SURVEY_QUESTIONS
from cag.io.llm import load_api_key

# Attribute maps and IDs
from gabm.abm.attributes.gender import GenderMap, GenderID
from gabm.abm.attributes.politics import PoliticsID
from gabm.abm.democracy.election import ElectionID
from cag.abm.attributes.region import UKRegionMap, RegionID
from cag.abm.attributes.education import SurveyEducationMap, EducationID
from cag.abm.attributes.ethnicity import SurveyEthnicityMap, EthnicityID
from cag.abm.attributes.income import SurveyIncomeMap, IncomeID
from cag.abm.attributes.politics import SurveyPoliticsMap
from cag.abm.attributes.family import SurveyFamilyMap, FamilyID
from cag.abm.democracy.elections.ukge2019 import UKGE2019VoteMap, UKGE2019VoteID
from cag.abm.democracy.elections.brexit import BrexitVoteMap, BrexitVoteID
from cag.abm.attributes.narratives import (
    SelftranscMap, SelfenhMap, OpennessMap, ConformTradMap, SDOMap, EDOMap, RWAMap,
    rescale_1_6, rescale_1_7,
)

print("Imports OK")

## 1. Load Data & Build Environment

In [ ]:
random.seed(42)
year = 2026

UKGE2019_ELECTION_ID = ElectionID(0)
BREXIT_REFERENDUM_ID = ElectionID(1)

sn = SurveyedNation(
    year=year, place="UK",
    gender_map=GenderMap(),
    region_map=UKRegionMap(),
    education_map=SurveyEducationMap(),
    ethnicity_map=SurveyEthnicityMap(),
    income_map=SurveyIncomeMap(),
    politics_map=SurveyPoliticsMap(),
    family_map=SurveyFamilyMap(),
    ukge2019_vote_map=UKGE2019VoteMap(UKGE2019_ELECTION_ID),
    brexit_vote_map=BrexitVoteMap(BREXIT_REFERENDUM_ID),
    selftransc_map=SelftranscMap,
    selfenh_map=SelfenhMap,
    openness_map=OpennessMap,
    conformtrad_map=ConformTradMap,
    sdo_map=SDOMap,
    edo_map=EDOMap,
    rwa_map=RWAMap,
)

data = load("../data/yougov_survey_data/YouGovProcessedData_train.csv")

for i in range(len(data)):
    row = data.iloc[i]
    sc = SurveyedCitizen(
        agent_id=row.get('ID', None), environment=sn,
        year_of_birth=year - int(row.get('age', 0)),
        gender_id=GenderID.MALE if int(row.get('male_dummy', 0)) == 1 else GenderID.FEMALE,
        region_id=RegionID(int(row.get('tprofile_GOR', 0))),
        education_id=EducationID(int(row.get('profile_education_level', 0))),
        income_id=IncomeID(int(row.get('tprofile_gross_household', 0))),
        ethnicity_id=EthnicityID(int(row.get('ethnicity_R', 0))),
        family_id=FamilyID.PARENT if int(row.get('parent_dummy', 0)) == 1 else FamilyID.NOT_PARENT,
        ukge2019_vote_id=UKGE2019VoteID(int(row.get('Vote2019R', 0))),
        brexit_vote_id=BrexitVoteID(int(row.get('pastvote_EURef', 0))),
        politics_id=PoliticsID(int(row.get('Political_Left_Right', 0))),
        selftransc_id=rescale_1_6(int(row.get('Selftransc_Val', 0))),
        selfenh_id=rescale_1_6(int(row.get('Selfenh_Values', 0))),
        openness_id=rescale_1_6(int(row.get('Openness', 0))),
        conformtrad_id=rescale_1_6(int(row.get('ConformTrad', 0))),
        sdo_id=rescale_1_7(int(row.get('SDO', 0))),
        edo_id=rescale_1_7(int(row.get('EDO', 0))),
        rwa_id=rescale_1_6(int(row.get('RWA', 0))),
        original_survey_data=data.iloc[i],
    )
    sn.agents_active[sc.id] = sc

sn.political_agent_a = PoliticalAgent("agent_a", "pro_climate")
sn.political_agent_b = PoliticalAgent("agent_b", "anti_climate")
sn.assign_political_exposure()
sn.create_network(seed=42)
sn.assign_network_blocks()

api_key = load_api_key("../data/api_key.csv")
target_policy = ClimatePolicyID.CARBON_TAX

n_total = len(sn.agents_active)
n_with_neighbors = sum(1 for c in sn.agents_active.values() if len(c.network_neighbors) > 0)
print(f"Total citizens: {n_total}")
print(f"Citizens with network neighbors: {n_with_neighbors}")

## 2. Run Phase C: Peer Conversation

Simultaneous update: ALL citizen messages are generated BEFORE any reflections occur.  
Each citizen talks to at most `k` random network neighbors (default: 3).

In [ ]:
# TODO: Implement after Issue 7 code is complete
# MAX_DEMO = 10
# result_c = sn.run_peer_conversation(
#     policy_id=target_policy, day=1, k_conversations=3,
#     api_key=api_key, model="gpt-4o-mini",
# )

## 3. Sample Peer Messages & Reflections

In [ ]:
# TODO: Show sample generated peer messages and resulting reflections

## 4. Simultaneous Update Verification

Verify that no citizen's reflection influenced another citizen's message within the same phase.

In [ ]:
# TODO: Verify simultaneous update property

## 5. Conversation Pattern Analysis

Distribution of conversations per citizen, neighbor overlap, and message word counts.

In [ ]:
# TODO: Analyze conversation patterns (messages sent/received per citizen, k distribution)

## 6. Sanity Checks

In [ ]:
# TODO: Sanity checks
# - Citizens with no neighbors produced no reflection
# - Each citizen talked to at most k neighbors
# - Reflections have phase="C" and messages_received contains actual peer messages
# - Total message count <= k * n_demo_citizens

## 7. LLM Call Summary

In [ ]:
# TODO: LLM call cost summary for Phase C